# Inference with a Hub model

Load the MOUSE model pushed by `02_train_offline.ipynb` from the Hugging Face Hub and run a few cached evaluation steps to verify the checkpoint works.

Expand the synthetic evaluation steps into a full environment loop for real rollouts.

In [ ]:
import torch
from tensordict import TensorDict

from mouse_core import load_model

# Hub model repo written by 02_train_offline.ipynb.
MODEL_ID = "mouse-example-model"

# Limit greedy action selection to FrozenLake's valid discrete actions.
MAX_ACTIONS = 4

# Number of cached one-step forwards to run as a smoke test.
EVAL_STEPS = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model(MODEL_ID, map_location="cpu").eval().to(device)

## Cached evaluation steps

Feed a few one-step `TensorDict`s through the loaded model, reuse the cache between calls, and print the greedy action at each step.

In [ ]:
cache = None
last_action = torch.zeros(1, 1, dtype=torch.long)

with torch.no_grad():
    for step_idx in range(EVAL_STEPS):
        step_stream = TensorDict(
            {
                "action": last_action,
                "reward": torch.zeros(1, 1, dtype=torch.float32),
                "done": torch.zeros(1, 1, dtype=torch.long),
            },
            batch_size=(1, 1),
        ).to(device)

        out, cache = model(step_stream, cache=cache, use_cache=True)
        action = model.get_action(out, temperature=0.0, num_actions=MAX_ACTIONS)
        q_values = out["action_value"][0, -1, :MAX_ACTIONS].detach().cpu()
        print(f"step={step_idx} action={action.item()} q_values={q_values.tolist()}")
        last_action = action.reshape(1, 1).cpu()